# SpikeGate: Dynamic Neuromorphic Hardware Gating Analysis
This notebook runs the full `spikegate` analysis pipeline on the `MaxFormer` Spiking Neural Network natively in Google Colab.

In [ ]:
!git clone https://github.com/SiddheshUttarwar/SNN-Transformer-Knowlege-Extraction.git
%cd SNN-Transformer-Knowlege-Extraction/MaxFormer

# Install core dependencies
!pip install datasets torchvision spikingjelly timm

## 1. Load Model Checkpoint
Ensure your `10-384-T4.pth.tar` checkpoint is uploaded to Google Drive, then mount your drive and copy it to the `checkpoints` directory.

In [ ]:
# Mount Google Drive if your checkpoint is there
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoints directory and copy the model
!mkdir -p checkpoints
!cp "/content/drive/MyDrive/10-384-T4.pth.tar" ./checkpoints/10-384-T4.pth.tar

# If your file path is different, adjust the cp command above!

from qkv_sparsity_extractor import load_maxformer
model, _, device = load_maxformer()
print(f"Model loaded on {device}")

## 2. Load Calibration Data
We use HuggingFace `datasets` to stream the ImageNet-1K validation set to avoid downloading the massive 150GB dataset into Colab.

In [ ]:
import torch
from datasets import load_dataset
from torchvision import transforms

IMG_SIZE = 224
BATCH_SIZE = 10

dataset = load_dataset('mrm8488/ImageNet1K-val', split='train', streaming=True, trust_remote_code=True)
tfs = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def data_generator():
    b = []
    for item in dataset:
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        b.append(tfs(img).unsqueeze(0))
        if len(b) == BATCH_SIZE:
            yield torch.cat(b, dim=0), torch.zeros(BATCH_SIZE)
            b = []

gen = data_generator()
calibration_data = [next(gen) for _ in range(5)] # 50 samples for quick profiling

## 3. Run SpikeGate AutoProfiler
Automatically discovers MaxFormer's Spatio-Temporal output shape `(T, B, C, N)`, dynamically ungroups the 6 heads, and extracts detailed timing, sparsity, and ghost/dead neuron percentages.

In [ ]:
from spikegate.profiler import AutoProfiler

# MaxFormer has T=4 timesteps and 6 heads per block
profiler = AutoProfiler(model, T=4)
profile_json = profiler.calibrate(
    calibration_data, 
    device=device, 
    num_blocks=10, 
    num_heads=6, 
    out_file="colab_gating_profile.json"
)

import json
print("Sample Policy for Block 0, Head 0:")
print(json.dumps(profile_json["block_0"]["head_0"], indent=2))

## 4. Run Head Importance Estimation
This runs the evaluator directly on the MaxFormer without modifying the base logic, using `MASK-ONE-OUT` and `ISOLATE-ONE` ablation strategies to identify the correlation between gating policies and actual model capability.

In [ ]:
from spikegate.gating import DynamicGateController
from spikegate.evaluator import GatingAblationStudy

gate_ctrl = DynamicGateController("colab_gating_profile.json")

# We pass the model directly. To test *Compute Savings*, you'd need the SpikingGatedAttention wrappers.
# But for Head Importance Evaluation, passing the raw model will work because the Evaluator uses 
# the Gate Controller's overrides to virtually "gate" the outputs if they were wrapped.
evaluator = GatingAblationStudy(model, gate_ctrl)

print("Fetching 5 batches for evaluation...")
eval_data = [next(gen) for _ in range(5)]

# Fast test on 2 blocks/6 heads. 
# Set num_blocks=10 to run the full massive study!
importance_report = evaluator.run_head_importance_study(
    eval_data, 
    device=device, 
    num_blocks=2, 
    num_heads=6, 
    out_file="colab_head_importance.json",
    checkpoint_file="colab_head_importance_checkpoint.json"
)

print("\nAggregated Insights (How much fidelity drops per policy group):")
print(json.dumps(importance_report.get("AGGREGATED_INSIGHTS", {}), indent=2))